In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import warnings 
warnings.filterwarnings("ignore")

In [3]:
from DFTStructureGenerator import B_N_Cl, mol_manipulation
import glob, os
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
from tqdm import tqdm
import pandas as pd
import pickle

In [26]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import KFold, cross_val_score
from catboost import CatBoostRegressor, CatBoostClassifier
from matplotlib import pyplot as plt
import seaborn as sns

In [5]:
# PhysOrg
all_descriptors_names = [
    "C_Cl react deltaG", "C_Cl Cl_charge", "C_Cl C_charge", "C_Cl Cl_Hirshfield_charge", "C_Cl C_Hirshfield_charge", 
    "C_Cl MaxOtherCharge", "C_Cl MinOtherCharge", "C_Cl MaxChargeDist", "C_Cl MinOtherChargeDist", 
    "C_Cl MaxOtherHCharge", "C_Cl MinOtherHCharge", "C_Cl MaxHChargeDist", "C_Cl MinHOtherChargeDist", 
    "C_Cl MaxAromaticRingCharge", "C_Cl MinAromaticRingCharge", "C_Cl MaxAromaticChargeDist", "C_Cl MinAromaticChargeDist",
    "C_Cl C_Cl_dist", 
    "C_Cl HOMO-1", "C_Cl HOMO", "C_Cl LUMO", "C_Cl LUMO+1", "C_Cl dipole", "C_Cl Bv2radius", "C_Cl Bv4radius", "C_Cl Bv6radius", "C C_charge", "C C_spin", "C C_Hirshfield_charge", 
    "C Alpha-HOMO-1", "C Alpha-HOMO", "C Alpha-LUMO", "C Alpha-LUMO+1", "C Beta-HOMO-1", "C Beta-HOMO", "C Beta-LUMO", "C Beta-LUMO+1", "C_dipole",
    
    "B_G", "B_HOMO-1", "B_HOMO", "B_LUMO", "B_LUMO+1", "B_HOMO-1_B", "B_HOMO_B", "B_LUMO_B", "B_LUMO+1_B", "B_dipole", "B_spin",
    # "B_Cl_G", "B_Cl_HOMO-1", "B_Cl_HOMO", "B_Cl_LUMO", "B_Cl_LUMO+1", "B_Cl_dipole", "B_Cl_Charge_B", "B_Cl_Charge_Cl", "B_Cl_Bond_B_Cl", "B_Cl_BV_2A", "B_Cl_BV_4A", "B_Cl_BV_6A",
    
    "Ini_G", "Ini_HOMO-1", "Ini_HOMO", "Ini_LUMO", "Ini_LUMO+1", "Ini_dipole", "Ini_T","Ini_Charge_N", "Ini_Charge_C", "Ini_Bond_N_C", "Ini_BV_2A", "Ini_BV_4A", "Ini_BV_6A",
    "Sol_G", "Sol_HOMO-1", "Sol_HOMO", "Sol_LUMO", "Sol_LUMO+1", "Sol_dipole", "Sol_x",
    "S_G", "S_HOMO-1", "S_HOMO", "S_LUMO", "S_LUMO+1", "S_dipole", "S_Charge_S", "S_Charge_C", "S_Bond_S_C", "S_BV_2A", "S_BV_4A", "S_BV_6A",
    "B_eq",
    ]
len(all_descriptors_names)

82

In [34]:
def draw_correlation_map(X, figure_size=(5, 5), colors='coolwarm', useSVG=False, save_name='test', annot=True, show_label=False):
    df = pd.DataFrame(X)
    correlation_matrix = np.abs(df.corr())
    mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
    print(np.max(np.nan_to_num(correlation_matrix.to_numpy()[~mask], 0)))
    
    f, ax = plt.subplots(figsize=figure_size, dpi=300)
    annot_kws = {"fontsize": 30}
    ax = sns.heatmap(correlation_matrix, 
            mask=mask,
            cmap='coolwarm',    
            annot=annot,         
            fmt='.1f',          
            center=0,           
            cbar=1,
            annot_kws=annot_kws,)
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=30)
    if not show_label:
        ax.set_xticklabels([])
        ax.set_yticklabels([])
    plt.tight_layout()
    if useSVG:
        plt.savefig(f"{save_name}.svg", format="svg", bbox_inches='tight')
    else:
        plt.savefig(f"{save_name}.png", dpi=300, bbox_inches='tight')

# 1.Load Descritpors

In [6]:
with open("Data/Cldes_new.pkl", 'rb') as f:
    Cl_des_map = pickle.load(f)

In [12]:
with open("Data/Fingerprint.pkl", 'rb') as f:
    index_map = pickle.load(f)

In [22]:
with open("Data/PhysOrgdes_new.pkl", 'rb') as f:
    index_map = pickle.load(f)

In [13]:
def descriptor_generator(csv_file):
    if type(csv_file) == str:
        result_csv = pd.read_excel(csv_file).dropna()
    else:
        result_csv = csv_file.dropna()
    all_X = []
    yields = []
    for row_id, row in result_csv.iterrows():
        desc = []
        B = int(row['B_Index'])
        B_eq = row['eqs']
        ini = int(row['ini_Index'])
        sol_1 = int(row['sol_Index'])
        S = int(row['S_Index'])
        if "Cl_Index" in result_csv.columns:
            Cl = int(row['Cl_Index'])
            Cl_atomid = int(row['Cl_Atomid'])
            if Cl in [5001, 5002, 5003, 5004]:
                desc += Cl_des_map[f"Cl_{Cl:05d}_Claid_{Cl_atomid:05d}"]
            else:
                desc += Cl_des_map[f"Cl_{Cl:05d}"]
        desc += index_map[B]
        # desc += index_map[B + 1000]
        desc += index_map[ini]
        desc += index_map[sol_1]
        desc += index_map[S]
        desc += [B_eq]
        # desc += [1 if each == B_eq else 0 for each in [1.5, 2, 2.5,3]]
        all_X.append(desc)
        yields.append(row['yield'])
    yields = np.array(yields)
    all_X_qm = np.array(all_X)
    return all_X_qm, yields


In [36]:
all_X_qm, yields = descriptor_generator("Data/Iteration/Result_sum_00022.xlsx")
all_X_qm.shape

(42, 44)

# 2. Feature Selection

In [ ]:
draw_correlation_map(all_X_qm, save_name='test', useSVG=1, figure_size=(5,5), annot=False, show_label=True)

# 3. Model Performance

## 3.1Regressor

In [24]:
kf = KFold(n_splits=5, shuffle=True, random_state=1)
all_X = all_X_qm
# all_X = np.concatenate([all_X_mf, all_X_qm], axis=1)
all_r2s, all_f1s, all_acc, all_preds = [], [], [], np.zeros(len(all_X))
result_matrix = np.zeros((2, 2))
target = yields
for train_ids, test_ids in kf.split(all_X):
    # model = RandomForestRegressor(n_estimators=200, random_state=0)
    # model = XGBRegressor(n_estimators=100, random_state=0)
    # model = GradientBoostingRegressor(n_estimators=200, random_state=0)
    model = CatBoostRegressor(iterations=10000, random_seed=0, verbose=0, depth=2)
    model.fit(all_X[train_ids], target[train_ids])
    pred = model.predict(all_X[test_ids])
    r2 = r2_score(target[test_ids], pred)
    all_r2s.append(r2)
    print("R2: ", r2)
    a = target[test_ids] > 20
    b = pred > 20
    r2 = accuracy_score(a,b)
    print("Accuracy: ", r2)
    all_acc.append(r2)
    r2 = f1_score(a,b)
    print("F1: ",r2)
    all_f1s.append(r2)
    result_matrix += confusion_matrix(a,b)
    all_preds[test_ids] = pred
print(np.mean(all_r2s))


R2:  0.6545483454957209
Accuracy:  0.8269230769230769
F1:  0.8615384615384616
R2:  0.7864946262816591
Accuracy:  0.8846153846153846
F1:  0.911764705882353
R2:  0.5167544083558675
Accuracy:  0.8627450980392157
F1:  0.9041095890410958
R2:  0.7914970423576853
Accuracy:  0.8627450980392157
F1:  0.9014084507042254
R2:  0.755700022397729
Accuracy:  0.9019607843137255
F1:  0.9295774647887323
0.7009988889777323


## 3.2 Classifier

In [27]:
kf = KFold(n_splits=5, shuffle=True, random_state=1)
all_X = all_X_qm
# all_X = np.concatenate([all_X_mf, all_X_qm], axis=1)
all_f1s, all_acc, all_preds = [], [], np.zeros(len(all_X))
result_matrix = np.zeros((2, 2))
target = np.array([1 if each > 20 else 0 for each in yields])
for train_ids, test_ids in kf.split(all_X):
    # model = RandomForestClassifier(n_estimators=200, random_state=0)
    # model = XGBClassifier(n_estimators=100, random_state=0)
    # model = GradientBoostingClassifier(n_estimators=200, random_state=0)
    model = CatBoostClassifier(iterations=10000, random_seed=0, verbose=0, depth=2)
    model.fit(all_X[train_ids], target[train_ids])
    pred = model.predict(all_X[test_ids])
    r2 = accuracy_score(target[test_ids], pred)
    print("Accuracy: ", r2)
    all_acc.append(r2)
    r2 = f1_score(target[test_ids], pred)
    print("F1: ",r2)
    all_f1s.append(r2)
    result_matrix += confusion_matrix(target[test_ids], pred)
    
    all_preds[test_ids] = pred
result_matrix = result_matrix / 4
print(np.mean(all_acc), np.mean(all_f1s))


Accuracy:  0.8269230769230769
F1:  0.8615384615384616
Accuracy:  0.9038461538461539
F1:  0.9253731343283582
Accuracy:  0.803921568627451
F1:  0.8571428571428571
Accuracy:  0.9019607843137255
F1:  0.927536231884058
Accuracy:  0.9215686274509803
F1:  0.9444444444444445
0.8716440422322774 0.9032070258676359


# 4. Feature Importance

## 4.1 Model Feature Importance

In [38]:
# PhysOrg
all_descriptors_names = [
    "C_Cl react deltaG", "C_Cl Cl_charge", "C_Cl C_charge", "C_Cl Cl_Hirshfield_charge", "C_Cl C_Hirshfield_charge", 
    "C_Cl MaxOtherCharge", "C_Cl MinOtherCharge", "C_Cl MaxChargeDist", "C_Cl MinOtherChargeDist", 
    "C_Cl MaxOtherHCharge", "C_Cl MinOtherHCharge", "C_Cl MaxHChargeDist", "C_Cl MinHOtherChargeDist", 
    "C_Cl MaxAromaticRingCharge", "C_Cl MinAromaticRingCharge", "C_Cl MaxAromaticChargeDist", "C_Cl MinAromaticChargeDist",
    "C_Cl C_Cl_dist", 
    "C_Cl HOMO-1", "C_Cl HOMO", "C_Cl LUMO", "C_Cl LUMO+1", "C_Cl dipole", "C_Cl Bv2radius", "C_Cl Bv4radius", "C_Cl Bv6radius", "C C_charge", "C C_spin", "C C_Hirshfield_charge", 
    "C Alpha-HOMO-1", "C Alpha-HOMO", "C Alpha-LUMO", "C Alpha-LUMO+1", "C Beta-HOMO-1", "C Beta-HOMO", "C Beta-LUMO", "C Beta-LUMO+1", "C_dipole",
    
    "B_G", "B_HOMO-1", "B_HOMO", "B_LUMO", "B_LUMO+1", "B_HOMO-1_B", "B_HOMO_B", "B_LUMO_B", "B_LUMO+1_B", "B_dipole", "B_spin",
    # "B_Cl_G", "B_Cl_HOMO-1", "B_Cl_HOMO", "B_Cl_LUMO", "B_Cl_LUMO+1", "B_Cl_dipole", "B_Cl_Charge_B", "B_Cl_Charge_Cl", "B_Cl_Bond_B_Cl", "B_Cl_BV_2A", "B_Cl_BV_4A", "B_Cl_BV_6A",
    
    "Ini_G", "Ini_HOMO-1", "Ini_HOMO", "Ini_LUMO", "Ini_LUMO+1", "Ini_dipole", "Ini_T","Ini_Charge_N", "Ini_Charge_C", "Ini_Bond_N_C", "Ini_BV_2A", "Ini_BV_4A", "Ini_BV_6A",
    "Sol_G", "Sol_HOMO-1", "Sol_HOMO", "Sol_LUMO", "Sol_LUMO+1", "Sol_dipole", "Sol_x",
    "S_G", "S_HOMO-1", "S_HOMO", "S_LUMO", "S_LUMO+1", "S_dipole", "S_Charge_S", "S_Charge_C", "S_Bond_S_C", "S_BV_2A", "S_BV_4A", "S_BV_6A",
    "B_eq",
    ]
len(all_descriptors_names)

82

In [39]:
all_X_qm, yields = descriptor_generator("Data/Iteration/Result_sum_00022.xlsx")
model = CatBoostRegressor(iterations=10000, random_seed=0, verbose=0, depth=2)
model.fit(all_X_qm, yields)
importance = model.feature_importances_
np.array(all_descriptors_names)[np.argsort(importance)[::-1]]


array(['B_spin', 'B_HOMO_B', 'B_HOMO-1', 'Sol_dipole', 'Sol_x',
       'Ini_HOMO', 'Ini_LUMO+1', 'C_Cl MinAromaticChargeDist',
       'C_Cl C_Hirshfield_charge', 'Ini_T', 'C_Cl C_charge',
       'C_Cl Bv4radius', 'B_eq', 'C_Cl react deltaG', 'C_Cl HOMO-1',
       'Sol_HOMO', 'C Beta-LUMO+1', 'Sol_HOMO-1',
       'C_Cl MinHOtherChargeDist', 'C_Cl MinAromaticRingCharge', 'Sol_G',
       'S_Charge_C', 'C_Cl dipole', 'C_Cl HOMO', 'C_Cl Bv6radius',
       'C Alpha-HOMO', 'Sol_LUMO+1', 'Ini_Charge_C', 'Ini_G',
       'C_Cl MaxHChargeDist', 'C_Cl MaxAromaticRingCharge',
       'Ini_Charge_N', 'S_Bond_S_C', 'C C_spin', 'C_Cl MaxOtherHCharge',
       'Sol_LUMO', 'C_Cl Bv2radius', 'C_dipole', 'C_Cl MinOtherHCharge',
       'C_Cl MaxAromaticChargeDist', 'Ini_BV_2A', 'Ini_BV_6A',
       'C_Cl MaxChargeDist', 'C C_charge', 'B_LUMO_B', 'S_dipole',
       'B_dipole', 'Ini_LUMO', 'B_LUMO+1_B', 'B_HOMO', 'B_HOMO-1_B',
       'B_G', 'S_HOMO-1', 'C_Cl MinOtherChargeDist', 'B_LUMO',
       'S_Charge_S', '

## 4.2 PYSR

In [ ]:
from pysr import PySRRegressor

model = PySRRegressor(
    maxsize=20,
    niterations=40,  # < Increase me for better results
    binary_operators=["+", "*"],
    unary_operators=[
        "cos",
        "exp",
        "sin",
        "inv(x) = 1/x",
        # ^ Custom operator (julia syntax)
    ],
    extra_sympy_mappings={"inv": lambda x: 1 / x},
    # ^ Define operator for SymPy as well
    elementwise_loss="loss(prediction, target) = (prediction - target)^2",
    # ^ Custom loss function (julia syntax)
)

In [ ]:
model.fit(all_X_qm[:,np.argsort(importance)[::-1][:15]], yields)

In [ ]:
plt.scatter(model.predict(all_X_qm[:,np.argsort(importance)[::-1][:15]], 13), yields)

In [ ]:
r2_score(model.predict(all_X_qm[:,np.argsort(importance)[::-1][:15]], 13), yields)

## 4.3 Feature Selection by Performance

In [ ]:
from copy import deepcopy
def feature_selection(X, y, train, test):
    remain_list = []
    all_list = list(range(len(X[0])))
    #removed_lists = Bayesian_Optimization.index_of_highly_correlated_features(X, threshold=0.98)
    for idx in range(len(X[0])):
        all_r2 = []
        for id, each in enumerate(all_list):
            print(id, len(all_list), end='\r')
            if each in remain_list:
                all_r2.append(-10)
                continue
            else:
                temp_remain_list = remain_list + [each]
            X_new = X[:, temp_remain_list]
            temp_r2 = []
            kfold = KFold(n_splits=5, shuffle=True, random_state=0)
            model = CatBoostRegressor(iterations=100, random_seed=0, verbose=0, depth=2)
            acc = cross_val_score(model, X_new[train], y[train], cv=kfold, n_jobs=1).mean()
            temp_r2 = acc.mean()
            all_r2.append(temp_r2)
        max_id = np.argmax(all_r2)
        if max_id in remain_list:
            remain_list.remove(max_id)
        else:
            remain_list.append(max_id)
        print(np.max(all_r2), remain_list)
    return all_r2, remain_list
all_r2, y_in_removed_lists = feature_selection(all_X_qm, yields, np.arange(len(all_X_qm)), [])